In [ ]:
#Installing kaggle library
#Uploading kaggle.json file (token to connect to kaggle)
!pip install kaggle
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"tanjaldir","key":"cb91ac02c819a45f54ad9446b7b44eb7"}'}

In [ ]:
#Moving kaggle.json file to correct location
#Setting appropriate permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#Downloading dataset
!kaggle datasets download -d tawsifurrahman/covid19-radiography-database -p /content/covid19_radiography

Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database
License(s): copyright-authors
100% 776M/778M [00:35<00:00, 25.6MB/s]
100% 778M/778M [00:35<00:00, 23.2MB/s]


In [ ]:
#Unzipping dataset
!unzip /content/covid19_radiography/covid19-radiography-database.zip -d /content/covid19_radiography

Streaming output truncated to the last 5000 lines.
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7921.png  
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7922.png  
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7923.png  
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7924.png  
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7925.png  
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7926.png  
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7927.png  
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7928.png  
  inflating: /content/covid19_radiography/COVID-19_Radiography_Dataset/Normal/masks/Normal-7929.png  
  inflating: /content/covid19_r

In [ ]:
#Importing libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from PIL import Image
import random

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input
from tensorflow.keras.applications import preprocess_input
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
#Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Setting base dir
base_dir = '/content/covid19_radiography/COVID-19_Radiography_Dataset'

In [ ]:
#Function to get image and mask paths as well as labels/target
def get_image_mask_paths(base_dir):
    images = []
    masks = []
    target = []

    #Looping through each case folder
    for folder in os.listdir(base_dir):
        case_dir = os.path.join(base_dir, folder)

        if os.path.isdir(case_dir):

          #Getting paths to images and masks
          image_dir = os.path.join(case_dir, 'images')
          mask_dir = os.path.join(case_dir, 'masks')

          #Looping through images
          for image in os.listdir(image_dir):
            image_path = os.path.join(image_dir, image)
            mask_path = os.path.join(mask_dir, image)

            images.append(image_path)
            masks.append(mask_path)
            target.append(folder)

    return images, masks, target

In [ ]:
#Getting image and mask paths
image_paths, mask_paths, target = get_image_mask_paths(base_dir)

In [ ]:
#Creating a dataframe to hold paths and labels
data = pd.DataFrame({
    'image_paths': image_paths,
    'mask_paths': mask_paths,
    'target': target
})

In [ ]:
#Splitting the dataset into training and validation sets
train_df, val_df = train_test_split(data, test_size = 0.2, stratify = data['target'], random_state = 42)

In [ ]:
#Creating a tensorflow dataset
train_dataset = tf.data.Dataset.from_tensor_slices((train_df['image_paths'].values, train_df['mask_paths'].values, train_df['target'].values))
val_dataset = tf.data.Dataset.from_tensor_slices((val_df['image_paths'].values, val_df['mask_paths'].values, val_df['target'].values))

In [ ]:
#Defining function to load and preprocess images and masks
def load_and_preprocess_image_and_mask(image_path, mask_path, target):
    #Loading the image and mask
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels = 3)
    image = tf.image.resize(image, [224, 224])

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels = 1)
    mask = tf.image.resize(mask, [224, 224])

    #Normalizing mask to be between 0 and 1
    mask = tf.cast(mask, tf.float32) / 255.0

    #Weighting the image with the mask
    weighted_image = image * mask

    return weighted_image, target

In [ ]:
#tf.data.AUTOTUNE is a tool in tensorflow that optimizes data input pipelines automatically, improving performance and resource utilization, adjusting the number of parallel calls
#Mapping the loading and preprocessing function
train_dataset = train_dataset.map(load_and_preprocess_image_and_mask, num_parallel_calls = tf.data.AUTOTUNE)
val_dataset = val_dataset.map(load_and_preprocess_image_and_mask, num_parallel_calls = tf.data.AUTOTUNE)

#.prefetch(tf.data.AUTOTUNE). This function allows the dataset to load the next batch of data while the current batch is being processed, improving data input efficiency.
train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
#Function EfficientNet preprocessing
def preprocess_for_efficientnet(image, label):
    image = preprocess_input(image)
    return image, label

In [ ]:
#Applying the preprocessing function to the dataset
train_dataset = train_dataset.map(preprocess_for_efficientnet)
val_dataset = val_dataset.map(preprocess_for_efficientnet)

In [ ]:
#Loading the EfficientNetB0 model with pre-trained weights
base_model = ResNet50(weights = 'imagenet', include_top = False, input_shape = (224, 224, 3))

#Adding a GlobalAveragePooling layer to reduce dimensionality
x = base_model.output
x = GlobalAveragePooling2D()(x)

#Defining the model that outputs features
feature_model = Model(inputs = base_model.input, outputs=x)

#Displaying model summary
feature_model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 299, 299, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_pad (ZeroPadding2D) │ (None, 305, 305, 3)    │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_conv (Conv2D)       │ (None, 150, 150, 64)   │          9,472 │ conv1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_bn                  │ (None, 150, 150, 64)   │            256 │ conv1_conv[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_relu (Activation)   │ (None, 150, 150, 64)   │              0 │ conv1_bn[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pad (ZeroPadding2D) │ (None, 152, 152, 64)   │              0 │ conv1_relu[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pool (MaxPooling2D) │ (None, 75, 75, 64)     │              0 │ pool1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_conv       │ (None, 75, 75, 64)     │          4,160 │ pool1_pool[0][0]       │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_bn         │ (None, 75, 75, 64)     │            256 │ conv2_block1_1_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_relu       │ (None, 75, 75, 64)     │              0 │ conv2_block1_1_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_conv       │ (None, 75, 75, 64)     │         36,928 │ conv2_block1_1_relu[0… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_bn         │ (None, 75, 75, 64)     │            256 │ conv2_block1_2_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_relu       │ (None, 75, 75, 64)     │              0 │ conv2_block1_2_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_0_conv       │ (None, 75, 75, 256)    │         16,640 │ pool1_pool[0][0]       │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_3_conv       │ (None, 75, 75, 256)    │         16,640 │ conv2_block1_2_relu[0… │
│ (Conv2D)                  │                        │                │                        │
├──────────────────────

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 23,534,592 (89.78 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [ ]:
#Function to extract features from the training dataset
def extract_features(dataset):
    features = []
    labels = []
    for images, targets in dataset:
        batch_features = feature_model(images)
        features.append(batch_features.numpy())
        labels.append(targets.numpy())
    return np.concatenate(features), np.concatenate(labels)

In [ ]:
#Extracting features and labels
X_train, y_train = extract_features(train_dataset)
X_val, y_val = extract_features(val_dataset)

In [ ]:
#Defining the order of classes
cases = ['Normal', 'COVID', 'Lung_Opacity', 'Viral Pneumonia']

#Creating a mapping from class names to integers
class_mapping = {case: idx for idx, case in enumerate(cases)}

#Function to decode and map labels
def decode_and_map_labels(labels):
    return np.array([class_mapping[label.decode('utf-8')] for label in labels])

#Decoding and mapping labels
y_train_encoded = decode_and_map_labels(y_train)
y_val_encoded = decode_and_map_labels(y_val)

#Printing results to verify
print("y_train:", y_train)
print("y_train_encoded:", y_train_encoded)
print("y_val:", y_val)
print("y_val_encoded:", y_val_encoded)

y_train: [b'Normal' b'COVID' b'Normal' ... b'Normal' b'Viral Pneumonia'
 b'Lung_Opacity']
y_train_encoded: [0 1 0 ... 0 3 2]
y_val: [b'Viral Pneumonia' b'Normal' b'Lung_Opacity' ... b'Normal' b'Normal'
 b'Normal']
y_val_encoded: [3 0 2 ... 0 0 0]


In [ ]:
np.unique(y_train_encoded)

array([0, 1, 2, 3])

In [ ]:
#Checking shape of data after pca
print(X_train.shape)
print(X_val.shape)

(16932, 2048)
(4233, 2048)


In [ ]:
#Initializing the scaler
scaler = StandardScaler()

#Fitting on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

#Transforming the test data using the same scaler
X_val_scaled = scaler.transform(X_val)

In [ ]:
#Checking shape of data after pca
print(X_train_scaled.shape)
print(X_val_scaled.shape)

(16932, 2048)
(4233, 2048)


In [ ]:
# SVM
###
import time
start_time = time.time()

#Instantiating SVM
clf2_name = "linear SVM"
clf2 = SVC(gamma = 0.01, kernel = "poly")

#Training the model on training data
clf2.fit(X_train_scaled, y_train_encoded)

#Making predictions on test set
y_pred = clf2.predict(X_val_scaled)

#Calculating accuracy
clf2_score = clf2.score(X_val_scaled, y_val_encoded)

#Calculating mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_val_encoded, y_pred, average = "macro")

#Measuring time
model2_time = (time.time() - start_time)/60

In [ ]:
#Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

#Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

#Show classification report
model2_cr = classification_report(y_val_encoded, y_pred)
print(model2_cr)

Model 2: --- 10.345757389068604 minutes ---
The score is: 0.8677061185920151
The mean F1-Score (unweighted) is: 0.8622813981203192
              precision    recall  f1-score   support

           0       0.87      0.94      0.90      2038
           1       0.88      0.73      0.80       723
           2       0.85      0.83      0.84      1203
           3       0.94      0.88      0.91       269

    accuracy                           0.87      4233
   macro avg       0.89      0.84      0.86      4233
weighted avg       0.87      0.87      0.87      4233



In [ ]:
# KNN
###
from sklearn import neighbors
import time
start_time = time.time()

#Instantiating classifier
clf3_name = "KNN"
clf3 = KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

#Training the model on training data
clf3.fit(X_train_scaled, y_train_encoded)

#Making predictions on test set
y_pred = clf3.predict(X_val_scaled)

#Calculating accuracys
clf3_score = clf3.score(X_val_scaled, y_val_encoded)

#Calculating mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_val_encoded, y_pred, average = "macro")

#Measuring time
model3_time = (time.time() - start_time)/60

In [ ]:
#Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

#Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

#Showing classification report
model3_cr = classification_report(y_val_encoded, y_pred)
print(model3_cr)

Model 3: --- 0.07792878150939941 minutes ---
The score is: 0.8006142215922514
The mean F1-Score (unweighted) is: 0.7808594594103151
              precision    recall  f1-score   support

           0       0.77      0.96      0.86      2038
           1       0.79      0.52      0.63       723
           2       0.83      0.70      0.76      1203
           3       0.98      0.80      0.88       269

    accuracy                           0.80      4233
   macro avg       0.84      0.74      0.78      4233
weighted avg       0.81      0.80      0.79      4233



In [ ]:
#Grad-CAM (Gradient-weighted Class Activation Mapping) technique to visualize which parts of an input image are important for a model’s predictions.
#Function to generate Grad-CAM heatmaps
def get_gradcam_heatmap(model, img_array, last_conv_layer_name):

    #The grad_model is created by specifying the model's inputs and outputs. The outputs include both the activations from the last convolutional layer
    #and the final model output (predictions). This allows us to compute the gradients of the model's predictions with respect to the convolutional layer's outputs.
    grad_model = tf.keras.models.Model(

        #last_conv_layer_name: The name of the last convolutional layer in the model. This activation map (output of this layer) is the basis for the generated heatmap.
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array)

        #Computing the index of the class that has the highest predicted probability for the input image.
        top_pred_index = tf.argmax(preds[0])

        #Extracting the predicted output for the class identified by top_pred_index from the model's predictions.
        top_class_channel = preds[:, top_pred_index]

    #Calculating the gradients of the loss with respect to the convolutional layer's outputs.
    grads = tape.gradient(top_class_channel, conv_outputs)

    #Computing the mean of the gradients across the spatial dimensions (height and width) for each filter (channel).
    #pooled_grads, will have a shape of (num_filters,), meaning it contains one value per filter representing its importance in the final prediction.
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    #Weighting the output feature map of the last conv layer by the gradients.
    #Extracting the feature map from the last convolutional layer's output.
    conv_outputs = conv_outputs[0]

    #Performing a matrix multiplication between the feature map (conv_outputs) and the pooled gradients (pooled_grads), and reshaping to add a new axis.
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]

    #Removing dimensions of size 1 from the tensor.
    heatmap = tf.squeeze(heatmap)

    #Calculating the mean across the last dimension (the channels) of the conv_outputs tensor (height, width, num_filters)
    #heatmap = K.mean(conv_outputs, axis=-1)

    #Applying a ReLU activation to remove negative values, ensuring that only the important regions are highlighted. Normalizing data.
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
#Function to display gradcam
def display_gradcam(img, heatmap, alpha=0.4):
    #Resizing heatmap to match the image size
    heatmap = np.array(Image.fromarray((heatmap * 255).astype(np.uint8)).resize((img.shape[1], img.shape[0])))

    #Applying the colormap
    colormap = plt.get_cmap('jet')

    #Normalizing heatmap to range [0, 1] for colormap application
    normalized_heatmap = heatmap / np.max(heatmap)

    #Applying the colormap
    colored_heatmap = colormap(normalized_heatmap)

    #Convert to RGB by discarding the alpha channel
    colored_heatmap = (colored_heatmap[..., :3] * 255).astype(np.uint8)

    #Converting original image to a PIL image for blending
    img_pil = Image.fromarray((img * 255).astype(np.uint8))

    #Creating a PIL image for the heatmap
    heatmap_image = Image.fromarray(colored_heatmap)

    #Blending heatmap and original image
    superimposed_img = Image.blend(img_pil, heatmap_image, alpha)

    #Normalizing for display
    return np.array(superimposed_img) / 255

In [ ]:
#Function to create a saliency map
def create_saliency_map(model, img_array):

    #Computing gradients with respect to the input image.
    #Createing a GradientTape context to record operations for automatic differentiation.
    with tf.GradientTape() as tape:

        #Watching the input image array so that we can compute gradients with respect to it.
        tape.watch(img_array)

        #Returning an array of probabilities for each class.
        preds = model(img_array)

        #Getting the predicted class index
        class_index = tf.argmax(preds[0])

        #Extracting the model's output probability for the predicted class.
        class_output = preds[0][class_index]

    #Computing the gradient of the predicted class output with respect to the input image.
    #The gradient tells how much a small change in each pixel value of the image would change the predicted class score.
    grads = tape.gradient(class_output, img_array)

    #Taking the absolute value of the gradients to focus on regions of high sensitivity.
    saliency = tf.abs(grads[0])

    #Normalizing the saliency map so that the values are between 0 and 1.
    saliency = (saliency - tf.reduce_min(saliency)) / (tf.reduce_max(saliency) - tf.reduce_min(saliency))
    return saliency.numpy()

In [ ]:
from google.colab import files

#Sample Grad-CAM and Saliency Maps for multiple cases
#Selecting 4 cases from the validation dataset
cases = ["Normal", "COVID", "Lung_Opacity", "Viral Pneumonia"]

#Setting a random seed for reproducibility
random.seed(42)

#Getting the total number of samples in the validation dataset
total_samples = sum(1 for _ in val_dataset)

#Randomly select 4 unique indices
random_indices = random.sample(range(total_samples), 4)

plt.figure(figsize=(10, 20))

for i, idx in enumerate(random_indices):
    #Selecting a single image and its label
    sample_img, sample_label = val_dataset.skip(i).take(1).get_single_element()

    #Removing the batch dimension to get a single image with shape (299, 299, 3)
    sample_img = sample_img[0]

    #Adding a new dimension to match the model's expected input shape (1, 299, 299, 3)
    sample_img = tf.expand_dims(sample_img, axis=0)

    #Getting the heatmap and prepare display
    heatmap = get_gradcam_heatmap(feature_model, sample_img, 'conv5_block3_out')
    gradcam_image = display_gradcam(sample_img[0].numpy(), heatmap)

    #Getting the saliency map
    saliency_map = create_saliency_map(feature_model, sample_img)

    ##Manually clip values to ensure they fall within [0, 1] for display
    gradcam_image = np.clip(gradcam_image, 0, 1)
    saliency_map = np.clip(saliency_map, 0, 1)

    # Plotting Grad-CAM, Heatmap, and Saliency Map side by side
    plt.subplot(4, 2, 2 * i + 1)
    plt.title(f"Grad-CAM: {cases[i]}")
    plt.imshow(gradcam_image)
    plt.axis('off')

    plt.subplot(4, 2, 2 * i + 2)
    plt.title(f"Saliency Map: {cases[i]}")
    plt.imshow(saliency_map, cmap='hot')
    plt.axis('off')

#Reducing spaces betweeen images in plot
plt.tight_layout()

#Saving plot
plt.savefig("gradcam_saliency_maps_masked.png", dpi=300, bbox_inches='tight')

#Downloading plot to local
files.download("gradcam_saliency_maps_masked.png")

plt.show()

Output hidden; open in https://colab.research.google.com to view.